# 1. bm25 retreiver 구현 테스트

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"
EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
DEVICE = "mps"   # 맥북이면 보통 "mps", 안되면 "cpu"
TOP_K = 5

# =========================
# 임베딩 클래스
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

# =========================
# 전체 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])

documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )

print(f"전체 문서 수: {len(docs)}")

# =========================
# BM25 Retriever 생성
# =========================
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = TOP_K

print("BM25 Retriever 생성 완료")

# =========================
# 테스트 질의
# =========================
query = "어린이 보호구역 제한속도는 얼마인가?"

results = bm25_retriever.invoke(query)

print(f"\n질문: {query}")
print(f"검색 결과 개수: {len(results)}\n")

for i, doc in enumerate(results, 1):
    print("=" * 100)
    print(f"[{i}] metadata")
    print(doc.metadata)
    print("-" * 100)
    print(doc.page_content[:1000])   # 너무 길면 앞부분만 보기
    print()

# 2. Dense vs bm25

## 2-1) 단일 질문

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"
EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
DEVICE = "mps"   # 안되면 "cpu"
TOP_K = 5

# =========================
# 임베딩 클래스
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

print("Chroma 연결 완료")

# =========================
# Dense Retriever 생성
# =========================
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
print("Dense Retriever 생성 완료")

# =========================
# BM25용 전체 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])

documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )

print(f"BM25용 전체 문서 수: {len(docs)}")

# =========================
# BM25 Retriever 생성
# =========================
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = TOP_K
print("BM25 Retriever 생성 완료")

# =========================
# 질의
# =========================
query = "어린이 보호구역 제한속도는 얼마인가?"

dense_results = dense_retriever.invoke(query)
bm25_results = bm25_retriever.invoke(query)

# =========================
# Dense 결과 출력
# =========================
print("\n" + "=" * 120)
print(f"[DENSE RESULTS] 질문: {query}")
print("=" * 120)

for i, doc in enumerate(dense_results, 1):
    print(f"\n[DENSE {i}] metadata")
    print(doc.metadata)
    print("-" * 120)
    print(doc.page_content[:1000])

# =========================
# BM25 결과 출력
# =========================
print("\n" + "=" * 120)
print(f"[BM25 RESULTS] 질문: {query}")
print("=" * 120)

for i, doc in enumerate(bm25_results, 1):
    print(f"\n[BM25 {i}] metadata")
    print(doc.metadata)
    print("-" * 120)
    print(doc.page_content[:1000])

## 2-2) 다수 질문

In [ ]:
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"
EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
DEVICE = "mps"   # 안되면 "cpu"
TOP_K = 5

# =========================
# 임베딩 클래스
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

print("Chroma 연결 완료")

# =========================
# Dense Retriever 생성
# =========================
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
print("Dense Retriever 생성 완료")

# =========================
# BM25용 전체 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])

documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )

print(f"BM25용 전체 문서 수: {len(docs)}")

# =========================
# BM25 Retriever 생성
# =========================
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = TOP_K
print("BM25 Retriever 생성 완료")

# =========================
# 질의
# =========================
queries = [
    "어린이 보호구역 제한속도는 얼마인가?",
    "스쿨존 속도위반 기준은?",
    "주정차 위반 과태료 기준은?",
    "음주운전 면허취소 기준은?",
    "불법주차 신고 가능 여부는?"
]

# =========================
# 결과 저장용
# =========================
comparison_results = {}
rows = []

# =========================
# 검색 및 저장
# =========================
for query in queries:
    dense_results = dense_retriever.invoke(query)
    bm25_results = bm25_retriever.invoke(query)

    comparison_results[query] = {
        "dense": [],
        "bm25": []
    }

    print("\n" + "=" * 120)
    print(f"질문: {query}")
    print("=" * 120)

    print("\n[DENSE TOP 3]")
    for i, doc in enumerate(dense_results[:3], 1):
        dense_item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["dense"].append(dense_item)

        rows.append({
            "query": query,
            "retriever": "dense",
            "rank": i,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

        print(f"\n[DENSE {i}]")
        print(doc.metadata)
        print(doc.page_content[:300])

    print("\n[BM25 TOP 3]")
    for i, doc in enumerate(bm25_results[:3], 1):
        bm25_item = {
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        }
        comparison_results[query]["bm25"].append(bm25_item)

        rows.append({
            "query": query,
            "retriever": "bm25",
            "rank": i,
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

        print(f"\n[BM25 {i}]")
        print(doc.metadata)
        print(doc.page_content[:300])

# =========================
# DataFrame 생성
# =========================
df_results = pd.DataFrame(rows)

print("\n저장 완료")
print(f"comparison_results 키 개수: {len(comparison_results)}")
print(f"df_results 행 개수: {len(df_results)}")

In [ ]:
df_results.to_csv("file/csv/dense_bm25_compare.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_results